<a href="https://colab.research.google.com/github/NBushey26/ds2002-fa26/blob/main/Copy_of_2026_09_25_%E2%80%94_Cleaning_Gauntlet_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


In [3]:
#Kept having to come back to try to calculate revenue, couldn't figure out how to do it in one stretch, does this work?
df_original = df.copy()
df_original["price"] = pd.to_numeric(
    df_original["price"].astype(str).str.replace("$", "", regex=False),
    errors="coerce"
)

df_original["qty"] = pd.to_numeric(df_original["qty"], errors="coerce")

original_revenue = (
    df_original["qty"].fillna(0) * df_original["price"]
).sum()

print("Original revenue:", round(original_revenue, 2))

Original revenue: 4852.5


### TODO 1 — drop duplicates

In [4]:
before = len(df)
df = df.drop_duplicates()
rows_affected = before - len(df)

log("Drop duplicates", "Removed duplicate rows", rows_affected)

print(df.shape)

[Drop duplicates] Removed duplicate rows (15 row(s))
(300, 5)


### TODO 2 — clean `price` -> float

In [5]:
df["price"] = pd.to_numeric(
    df["price"].astype(str).str.replace("$", "", regex=False),
    errors="coerce"
)

log("Clean price", "Removed dollar signs and converted price to float", len(df))

[Clean price] Removed dollar signs and converted price to float (300 row(s))


In [6]:
print("Revenue after duplicates:", (df["qty"] * df["price"]).sum())

Revenue after duplicates: 4594.5


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [7]:
df["qty"] = pd.to_numeric(df["qty"], errors="coerce")

bad_qty = df["qty"].isna() | (df["qty"] < 0)
rows_affected = bad_qty.sum()

df = df[~bad_qty].copy()

log("Clean qty", "Dropped rows with missing or negative qty", rows_affected)

[Clean qty] Dropped rows with missing or negative qty (25 row(s))


In [8]:
df["revenue"] = df["qty"] * df["price"]

print("Revenue by category:")
print(
    df.groupby("category")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .round(2)
)

print("Overall revenue:")
print("$" + format(df["revenue"].sum(), ".2f"))

Revenue by category:
category
Food         895.5
rain-gear    877.5
Merch        856.5
food         760.5
Apparel      715.5
RainGear     634.5
Name: revenue, dtype: float64
Overall revenue:
$4740.00


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [9]:
print(df["item"].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64


In [10]:
ITEM_MAP = {
    "Cheeseburger": "Cheeseburger",
    "cheese burger": "Cheeseburger",
    "Foam Finger": "Foam Finger",
    "foam finger": "Foam Finger",
    "Rain Poncho": "Rain Poncho",
    "rain poncho": "Rain Poncho"
}

before_unique = df["item"].nunique()

df["item"] = df["item"].map(ITEM_MAP)

after_unique = df["item"].nunique()

log(
    "Canonicalize item",
    "Collapsed alternate spellings into canonical product names",
    before_unique - after_unique
)

print(df["item"].value_counts())

[Canonicalize item] Collapsed alternate spellings into canonical product names (3 row(s))
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [11]:
CATEGORY_MAP = {
    "Food": "Food",
    "food": "Food",
    "Merch": "Merch",
    "Apparel": "Merch",
    "RainGear": "RainGear",
    "rain-gear": "RainGear"
}

before_unique = df["category"].nunique()

df["category"] = df["category"].map(CATEGORY_MAP)

after_unique = df["category"].nunique()

log("Normalize category",
    "Standardized category names and combined Apparel with Merch",
    before_unique - after_unique
)

print(df["category"].value_counts())

[Normalize category] Standardized category names and combined Apparel with Merch (3 row(s))
category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [11]:
assert df.duplicated().sum() == 0
assert df["qty"].min() >= 1
assert df["price"].dtype == float
assert set(df["item"].unique()) == {
    "Cheeseburger",
    "Foam Finger",
    "Rain Poncho"
}
assert set(df["category"].unique()) == {
    "Food",
    "Merch",
    "RainGear"
}
print("clean:", df.shape)

clean: (275, 6)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [12]:
df["revenue"] = df["qty"] * df["price"]

revenue_by_category = (
    df.groupby("category")["revenue"]
      .sum()
      .sort_values(ascending=False)
      .round(2)
)

print("Revenue by category:")
print(revenue_by_category)

print("Overall revenue:")
print(round(df["revenue"].sum(), 2))

Revenue by category:
category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
Overall revenue:
4740.0


**What I would tell the vendor:** I would tell the cendor to continue investing evenly between all off the categories because they are each making roughly the same amount in revenue.

### TODO 8 — read back your log

In [13]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,Drop duplicates,Removed duplicate rows,15
1,Clean price,Removed dollar signs and converted price to float,300
2,Clean qty,Dropped rows with missing or negative qty,25
3,Canonicalize item,Collapsed alternate spellings into canonical p...,3
4,Normalize category,Standardized category names and combined Appar...,3


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a. Removing duplicates seems to change my total revenue the most from 4852 dollars down to 4594 which is a 258 dollar difference. I had a lot of trouble trying to figure out when to calculate revenue with it starting as a string.

b.A reasonable person could have kept merch and apparel seperate instead of combining them the way I did. I know in the quesiton it said to decide and mentioned it was a business decision, but I thought it could be helpful to simplify. This wouldn't have changed anything with the revenue but it would have changed the information that we were recieving. I did this just to simplify the costs and the numbers but if kept seperate it could further breakdown what was selling because merch can be items other than just clothes.